In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("ModelTraining") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("Spark session started")

Spark session started


In [2]:
from google.colab import files
files.upload()

Saving final_data.zip to final_data.zip


In [3]:
import zipfile
print("\nEXTRACTING DATA\n")
zip_path = "/content/final_data.zip"
extract_path = "/content/final_data"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)
print("Data extracted successfully")


EXTRACTING DATA

Data extracted successfully


In [4]:
import os
print("\nVERIFYING DATA\n")
print(os.listdir("/content/final_data"))


VERIFYING DATA

['val', 'test', 'train']


In [5]:
print("\nLOADING DATA INTO SPARK\n")
train_ready = spark.read.parquet("/content/final_data/train/")
val_ready   = spark.read.parquet("/content/final_data/val/")
test_ready  = spark.read.parquet("/content/final_data/test/")
print("Train:", train_ready.count())
print("Validation:", val_ready.count())
print("Test:", test_ready.count())


LOADING DATA INTO SPARK

Train: 1447502
Validation: 361295
Test: 451689


In [6]:
from pyspark.ml.classification import RandomForestClassifier
print("\nTRAINING FINAL MODEL\n")
rf = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    numTrees=50,
    maxDepth=10,
    maxBins=200
)
rf_model = rf.fit(train_ready)
rf_preds = rf_model.transform(test_ready)
print("Model trained successfully")


TRAINING FINAL MODEL

Model trained successfully


In [7]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)
evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)
acc = evaluator_acc.evaluate(rf_preds)
f1 = evaluator_f1.evaluate(rf_preds)
print("Accuracy:", acc)
print("F1 Score:", f1)

Accuracy: 0.9381565634761971
F1 Score: 0.9169635346932142


In [8]:
print("\nSAVING MODEL\n")
rf_model.save("/content/rf_model_final")
print("Model saved successfully")


SAVING MODEL

Model saved successfully


In [14]:
print("\nSAVING PREDICTIONS\n")
rf_preds.select("label", "prediction") \
    .toPandas() \
    .to_csv("/content/final_predictions.csv", index=False)
print("Predictions saved successfully")


SAVING PREDICTIONS

Predictions saved successfully


In [16]:
import os
print(os.listdir("/content"))

['.config', 'final_data', 'rf_model_final', 'final_predictions.csv', 'final_data.zip', 'sample_data']


In [24]:
import os

print("Content folder:")
print(os.listdir("/content"))

print("\nEvaluation folder:")
if os.path.exists("/content/evaluation_output"):
    print(os.listdir("/content/evaluation_output"))

Content folder:
['.config', 'final_data', 'rf_model_final', 'model_predictions.csv', 'confusion_matrix.csv', 'final_predictions.csv', 'final_data.zip', 'sample_data']

Evaluation folder:


In [25]:
from google.colab import files
print("\nDOWNLOADING FILE\n")
files.download("/content/final_predictions.csv")


DOWNLOADING FILE



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [26]:
rf_preds.select(
    "label",
    "prediction",
    "State",
    "Weather_Condition",
    "hour"
).toPandas().to_csv("/content/model_predictions.csv", index=False)
print("Saved: model_predictions.csv")

Saved: model_predictions.csv


In [27]:
conf_matrix = rf_preds.groupBy("label", "prediction").count()
conf_matrix.toPandas().to_csv("/content/confusion_matrix.csv", index=False)
print("Saved: confusion_matrix.csv")

Saved: confusion_matrix.csv


In [29]:
import pandas as pd
feature_cols = [
    "Distance(mi)", "Temperature(F)", "Humidity(%)",
    "Pressure(in)", "Visibility(mi)", "Wind_Speed(mph)",
    "hour", "day_of_week", "month", "duration_minutes",
    "State_index", "Weather_Condition_index"
]
importances = rf_model.featureImportances
feature_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": list(importances)
}).sort_values(by="importance", ascending=False)
feature_importance.to_csv("/content/feature_importance.csv", index=False)
print("Saved: feature_importance.csv")

Saved: feature_importance.csv


In [31]:
import os
import shutil
import pandas as pd
from pyspark.sql import functions as F
total_rows = train_ready.count()

null_counts = train_ready.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in train_ready.columns
])
null_pd = null_counts.toPandas().T.reset_index()
null_pd.columns = ["column", "null_count"]
null_pd["percentage"] = (null_pd["null_count"] / total_rows) * 100
null_pd.to_csv("/content/data_quality_missing.csv", index=False)
severity = train_ready.groupBy("label").count()
severity.toPandas().to_csv("/content/severity_distribution.csv", index=False)

rf_preds.select("label", "prediction") \
    .toPandas() \
    .to_csv("/content/model_predictions.csv", index=False)
cm = rf_preds.groupBy("label", "prediction").count()
cm.toPandas().to_csv("/content/confusion_matrix.csv", index=False)
feature_cols = [
    "Distance(mi)", "Temperature(F)", "Humidity(%)",
    "Pressure(in)", "Visibility(mi)", "Wind_Speed(mph)",
    "hour", "day_of_week", "month", "duration_minutes",
    "State_index", "Weather_Condition_index"
]
importances = rf_model.featureImportances
fi = pd.DataFrame({
    "feature": feature_cols,
    "importance": list(importances)
}).sort_values(by="importance", ascending=False)
fi.to_csv("/content/feature_importance.csv", index=False)
train_ready.groupBy("State").count() \
    .toPandas().to_csv("/content/accidents_by_state.csv", index=False)
train_ready.groupBy("hour").count() \
    .toPandas().to_csv("/content/accidents_by_hour.csv", index=False)
train_ready.groupBy("Weather_Condition").count() \
    .toPandas().to_csv("/content/accidents_by_weather.csv", index=False)
zip_folder = "/content/tableau_files"
os.makedirs(zip_folder, exist_ok=True)
files_list = [
    "data_quality_missing.csv",
    "severity_distribution.csv",
    "model_predictions.csv",
    "confusion_matrix.csv",
    "feature_importance.csv",
    "accidents_by_state.csv",
    "accidents_by_hour.csv",
    "accidents_by_weather.csv"
]
for f in files_list:
    shutil.copy(f"/content/{f}", zip_folder)
shutil.make_archive("/content/Tableau_Zip", 'zip', zip_folder)
from google.colab import files
files.download("/content/Tableau_Zip.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>